### Downloading and Extracting the Caltech-256 Dataset

First, we'll download the Caltech-256 dataset, which is a collection of images of 256 object categories. Then, we'll extract the contents of the downloaded archive.

### Environment Setup
This section verifies the Python environment, including versions of PyTorch, Torchvision, and checks for CUDA availability and GPU details. This ensures the environment is properly configured for deep learning tasks.

In [ ]:
import sys
import torch
import torchvision
import platform

print("=== Environment Check ===")
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("System:", platform.platform())

print("\n=== CUDA Check ===")
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )
else:
    print("❌ GPU is NOT enabled")

=== Environment Check ===
Python: 3.13.15
PyTorch: 2.11.0+cu128
Torchvision: 0.26.0+cu128
System: Linux-6.6.122+-x86_64-with-glibc2.39

=== CUDA Check ===
CUDA available: True
GPU: Tesla T4
CUDA version: 12.8
GPU memory: 14.56 GB


### Mount Google Drive
This step mounts your Google Drive to the Colab environment, allowing the notebook to access and save files directly to your Drive. This is crucial for persistent storage of data, models, and experiment logs.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


### Define Project Structure
Here, we define a structured directory layout within Google Drive for organizing various project assets like data, model checkpoints, experiment results, and MLflow tracking files. This promotes a clean and manageable project workspace.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/vector_ai_track1")

folders = {
    "data": PROJECT_ROOT / "data",
    "splits": PROJECT_ROOT / "splits",
    "models": PROJECT_ROOT / "models",
    "checkpoints": PROJECT_ROOT / "checkpoints",
    "results": PROJECT_ROOT / "results",
    "mlruns": PROJECT_ROOT / "mlruns",
    "logs": PROJECT_ROOT / "logs",
}

for folder in folders.values():
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:")
print(PROJECT_ROOT)

print("\nFolders:")
for name, path in folders.items():
    print(f"{name:12} -> {path}")

Project root:
/content/drive/MyDrive/vector_ai_track1

Folders:
data         -> /content/drive/MyDrive/vector_ai_track1/data
splits       -> /content/drive/MyDrive/vector_ai_track1/splits
models       -> /content/drive/MyDrive/vector_ai_track1/models
checkpoints  -> /content/drive/MyDrive/vector_ai_track1/checkpoints
results      -> /content/drive/MyDrive/vector_ai_track1/results
mlruns       -> /content/drive/MyDrive/vector_ai_track1/mlruns
logs         -> /content/drive/MyDrive/vector_ai_track1/logs


### Install Required Libraries
This cell installs all necessary Python packages that are not pre-installed in the Colab environment. We include `ray[tune]` for hyperparameter tuning, `mlflow` for experiment tracking, `scikit-learn` for utilities, `pandas` for data handling, `matplotlib` for plotting, and `psutil` for system monitoring.

In [ ]:
!pip install -q "ray[tune]" mlflow scikit-learn pandas matplotlib psutil

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 112.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 120.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 93.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.7/148.7 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

### Verify Library Imports and Versions
After installation, we import the key libraries and print their versions. This step confirms that the installations were successful and helps in debugging potential version-related compatibility issues.

In [ ]:
import ray
import mlflow
import sklearn
import pandas as pd
import matplotlib
import psutil

print("Ray:", ray.__version__)
print("MLflow:", mlflow.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Pandas:", pd.__version__)
print("Matplotlib:", matplotlib.__version__)
print("psutil:", psutil.__version__)

Ray: 2.58.0
MLflow: 3.16.1
Scikit-learn: 1.6.1
Pandas: 2.2.3
Matplotlib: 3.10.0
psutil: 5.9.5


### Initialize Ray
Ray is an open-source framework that provides a simple, universal API for building distributed applications. Here, it's initialized to enable distributed computing functionalities, such as parallel data processing or hyperparameter tuning, specifying the number of CPUs and GPUs to use.

In [ ]:
import ray

ray.shutdown()

ray.init(
    num_cpus=2,
    num_gpus=1,
    include_dashboard=False,
    ignore_reinit_error=True
)

print("Ray initialized successfully")
print("Available resources:")
print(ray.available_resources())

2026-09-24 17:07:40,202	INFO worker.py:2024 -- Started a local Ray instance.


Ray initialized successfully
Available resources:
{'node:__internal_head__': 1.0, 'CPU': 2.0, 'object_store_memory': 3977587507.0, 'node:172.28.0.12': 1.0, 'memory': 9281037517.0, 'accelerator_type:T4': 1.0, 'GPU': 1.0}


### Configure MLflow Tracking
MLflow is an open-source platform for managing the end-to-end machine learning lifecycle. This section sets up MLflow to track experiments, specifying a SQLite database for tracking and a Google Drive folder for storing artifacts. This allows for organized logging and reproducibility of experiments.

In [ ]:
from pathlib import Path
import mlflow
from mlflow.tracking import MlflowClient

# MLflow paths on Google Drive
MLFLOW_DB = PROJECT_ROOT / "mlflow.db"
MLFLOW_ARTIFACTS = PROJECT_ROOT / "mlflow_artifacts"

MLFLOW_ARTIFACTS.mkdir(parents=True, exist_ok=True)

# SQLite tracking backend
TRACKING_URI = f"sqlite:///{MLFLOW_DB}"

mlflow.set_tracking_uri(TRACKING_URI)

print("Tracking URI:")
print(mlflow.get_tracking_uri())

print("\nDatabase:")
print(MLFLOW_DB)

print("\nArtifacts:")
print(MLFLOW_ARTIFACTS)

Tracking URI:
sqlite:////content/drive/MyDrive/vector_ai_track1/mlflow.db

Database:
/content/drive/MyDrive/vector_ai_track1/mlflow.db

Artifacts:
/content/drive/MyDrive/vector_ai_track1/mlflow_artifacts


### Create or Load MLflow Experiment
An MLflow experiment is a primary unit of organization for runs. This code either creates a new experiment with a specified name and artifact location or retrieves an existing one. All subsequent MLflow runs will be associated with this experiment.

In [ ]:
EXPERIMENT_NAME = "caltech256_track1"

client = MlflowClient()

experiment = client.get_experiment_by_name(EXPERIMENT_NAME)

if experiment is None:
    experiment_id = client.create_experiment(
        name=EXPERIMENT_NAME,
        artifact_location=MLFLOW_ARTIFACTS.as_uri()
    )
    print("✅ Experiment created")
    print("Experiment ID:", experiment_id)
else:
    experiment_id = experiment.experiment_id
    print("✅ Experiment already exists")
    print("Experiment ID:", experiment_id)

mlflow.set_experiment(EXPERIMENT_NAME)

✅ Experiment already exists
Experiment ID: 1


<Experiment: artifact_location='file:///content/drive/MyDrive/vector_ai_track1/mlflow_artifacts', creation_time=1789669105899, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1789669105899, lifecycle_stage='active', name='caltech256_track1', tags={}, trace_location=None, workspace='default'>

### Download and Load Caltech-256 Dataset
This cell downloads the Caltech-256 dataset, a large dataset of images categorized into 256 object classes. The dataset is downloaded to a local directory for efficient access during training and evaluation.

In [ ]:
from torchvision.datasets import Caltech256
from pathlib import Path

LOCAL_DATA_ROOT = Path("/content/datasets")
LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)

dataset = Caltech256(
    root=str(LOCAL_DATA_ROOT),
    download=True
)

print("✅ Caltech-256 downloaded")
print("Dataset length:", len(dataset))

100%|██████████| 1.18G/1.18G [00:07<00:00, 153MB/s]


✅ Caltech-256 downloaded
Dataset length: 30607


### Identify and Exclude Clutter Classes
The Caltech-256 dataset includes a 'clutter' category which typically consists of background images or images that do not belong to any specific object class. This step identifies and filters out these clutter classes to ensure that our model is trained only on relevant object categories.

In [ ]:
clutter_classes = [
    (i, name)
    for i, name in enumerate(dataset.categories)
    if "clutter" in name.lower()
]

print("Clutter classes found:", clutter_classes)

Clutter classes found: [(256, '257.clutter')]


### Filter Out Clutter Images
After identifying the clutter classes, this code creates a list of `valid_indices` corresponding to images that are part of actual object categories. This filtered list will be used to create subsets of the dataset for training, validation, and testing, excluding any clutter.

In [ ]:
import numpy as np

clutter_label_ids = {i for i, _ in clutter_classes}

valid_indices = [
    i
    for i, label in enumerate(dataset.y)
    if label not in clutter_label_ids
]

valid_labels = np.array([
    dataset.y[i]
    for i in valid_indices
])

print("Original dataset size:", len(dataset))
print("Object-only images:", len(valid_indices))
print("Number of object classes:", len(np.unique(valid_labels)))

Original dataset size: 30607
Object-only images: 29780
Number of object classes: 256


### Stratified Data Splitting
To evaluate the model's performance robustly, the dataset is split into training, validation, and test sets. A stratified split ensures that the proportion of each class is maintained across all subsets, preventing skewed distributions. The data is split into 70% for training, 15% for validation, and 15% for testing.

In [ ]:
from sklearn.model_selection import train_test_split

SEED = 42

# First split: 70% train, 30% temporary
train_indices, temp_indices = train_test_split(
    valid_indices,
    test_size=0.30,
    random_state=SEED,
    stratify=valid_labels
)

# Labels for the temporary split
temp_labels = np.array([
    dataset.y[i]
    for i in temp_indices
])

# Second split: 15% validation, 15% test
val_indices, test_indices = train_test_split(
    temp_indices,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_labels
)

print("Train:", len(train_indices))
print("Validation:", len(val_indices))
print("Test:", len(test_indices))

print("\nTotal:",
      len(train_indices) +
      len(val_indices) +
      len(test_indices))

Train: 20846
Validation: 4467
Test: 4467

Total: 29780


### Validate Data Splits and Save
This crucial step performs several checks:
1.  **Data Leakage Check:** Ensures there's no overlap between the train, validation, and test sets.
2.  **Class Coverage Check:** Verifies that all 256 object classes are present in each split, which is important for comprehensive model training and evaluation.
3.  **Class Distribution Summary:** Provides minimum and maximum image counts per class to understand balance.
4.  **Save Split Information:** The indices for each split (train, validation, test) along with other metadata are saved to a JSON file on Google Drive for reproducibility.

In [ ]:
import json
from collections import Counter

# =========================
# 1) Check data leakage
# =========================

train_set = set(train_indices)
val_set = set(val_indices)
test_set = set(test_indices)

print("=== Leakage Check ===")
print("Train ∩ Val :", len(train_set & val_set))
print("Train ∩ Test:", len(train_set & test_set))
print("Val ∩ Test  :", len(val_set & test_set))

assert len(train_set & val_set) == 0
assert len(train_set & test_set) == 0
assert len(val_set & test_set) == 0

print("✅ No data leakage")


# =========================
# 2) Check all 256 classes
# =========================

train_labels = [dataset.y[i] for i in train_indices]
val_labels = [dataset.y[i] for i in val_indices]
test_labels = [dataset.y[i] for i in test_indices]

print("\n=== Class Check ===")
print("Train classes     :", len(set(train_labels)))
print("Validation classes:", len(set(val_labels)))
print("Test classes      :", len(set(test_labels)))

assert len(set(train_labels)) == 256
assert len(set(val_labels)) == 256
assert len(set(test_labels)) == 256

print("✅ All 256 classes exist in every split")


# =========================
# 3) Show class distribution
# =========================

train_counts = Counter(train_labels)
val_counts = Counter(val_labels)
test_counts = Counter(test_labels)

print("\n=== Images per Class ===")
print(
    "Train min/max     :",
    min(train_counts.values()),
    max(train_counts.values())
)

print(
    "Validation min/max:",
    min(val_counts.values()),
    max(val_counts.values())
)

print(
    "Test min/max      :",
    min(test_counts.values()),
    max(test_counts.values())
)


# =========================
# 4) Save split to Google Drive
# =========================

split_file = folders["splits"] / "caltech256_split_seed42.json"

split_data = {
    "dataset": "Caltech-256",
    "seed": SEED,
    "num_classes": 256,
    "split_ratio": {
        "train": 0.70,
        "validation": 0.15,
        "test": 0.15
    },
    "excluded_classes": [
        name for _, name in clutter_classes
    ],
    "train_indices": [int(i) for i in train_indices],
    "val_indices": [int(i) for i in val_indices],
    "test_indices": [int(i) for i in test_indices]
}

with open(split_file, "w") as f:
    json.dump(split_data, f, indent=2)

print("\n=== Saved ===")
print("Split file:", split_file)
print("Exists:", split_file.exists())


# =========================
# 5) Final summary
# =========================

print("\n=== Final Split Summary ===")
print("Train      :", len(train_indices))
print("Validation :", len(val_indices))
print("Test       :", len(test_indices))

print("\n✅ Dataset preparation finished")

=== Leakage Check ===
Train ∩ Val : 0
Train ∩ Test: 0
Val ∩ Test  : 0
✅ No data leakage

=== Class Check ===
Train classes     : 256
Validation classes: 256
Test classes      : 256
✅ All 256 classes exist in every split

=== Images per Class ===
Train min/max     : 56 560
Validation min/max: 12 120
Test min/max      : 12 120

=== Saved ===
Split file: /content/drive/MyDrive/vector_ai_track1/splits/caltech256_split_seed42.json
Exists: True

=== Final Split Summary ===
Train      : 20846
Validation : 4467
Test       : 4467

✅ Dataset preparation finished


### Initial Test Cell
This is a placeholder cell for any preliminary testing or quick checks the user might want to perform during development. It currently contains a simple print statement.

In [ ]:
Test 1

SyntaxError: invalid syntax (1324977332.py, line 1)

### Initial Image Preprocessing Transforms (Attempt 1)
This cell defines the initial image preprocessing transformations using `torchvision.transforms`. These transforms include common augmentation techniques like random resizing and horizontal flipping for training, and resizing and center cropping for evaluation, followed by conversion to tensor and normalization with ImageNet mean and standard deviation. **Note:** This initial attempt did not explicitly handle images that might not be in RGB format, leading to a runtime error in subsequent steps.

In [ ]:
from torchvision import transforms

# ImageNet normalization values
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

# =========================
# Training preprocessing
# =========================

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=MEAN,
        std=STD
    )
])


# =========================
# Validation/Test preprocessing
# =========================

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=MEAN,
        std=STD
    )
])


print("✅ Preprocessing ready")

print("\nTrain transform:")
print(train_transform)

print("\nValidation/Test transform:")
print(eval_transform)

### Initial Dataset and DataLoader Setup (Attempt 1)
This section was an initial attempt to create `Caltech256` datasets and `Subset` datasets for training, validation, and testing using the transforms defined in the previous cell. The `download=False` argument is used because the dataset has already been downloaded. **Note:** This setup implicitly relied on all images being in a consistent format (e.g., RGB), which was not always the case, leading to errors during data loading.

In [ ]:
import torch
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import Caltech256

# =========================
# Dataset with train transform
# =========================

train_full_dataset = Caltech256(
    root=str(LOCAL_DATA_ROOT),
    download=False,
    transform=train_transform
)

# =========================
# Dataset with eval transform
# =========================

eval_full_dataset = Caltech256(
    root=str(LOCAL_DATA_ROOT),
    download=False,
    transform=eval_transform
)

# =========================
# Apply our fixed indices
# =========================

train_dataset = Subset(
    train_full_dataset,
    train_indices
)

val_dataset = Subset(
    eval_full_dataset,
    val_indices
)

test_dataset = Subset(
    eval_full_dataset,
    test_indices
)

print("Train images     :", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Test images      :", len(test_dataset))

### Initial DataLoader Creation (Attempt 1)
This cell was the initial attempt to create `DataLoader` objects for the training, validation, and test datasets. DataLoaders are responsible for batching and loading data efficiently. Parameters like `batch_size`, `shuffle`, `num_workers`, and `pin_memory` are set to optimize data loading. **Note:** This cell, when executed with the previous dataset setup, led to a `RuntimeError` because some images were not in a compatible format for the defined transformations.

In [ ]:
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("✅ DataLoaders created")

### Error Encountered during Data Loading
This cell attempted to fetch the first batch of images and labels from the `train_loader`. It resulted in a `RuntimeError` with the message: `output with shape [1, 224, 224] doesn't match the broadcast shape [3, 224, 224]`. This error indicates that some images were grayscale (single channel) while the normalization expected a 3-channel (RGB) image, causing a mismatch during the `Normalize` transform. This issue is addressed in the subsequent corrected preprocessing steps.

In [ ]:
images, labels = next(iter(train_loader))

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)

print("Images dtype:", images.dtype)
print("Labels dtype:", labels.dtype)

print("\nFirst 10 labels:")
print(labels[:10])

### Corrected Image Preprocessing Transforms (Attempt 2)
This cell provides the corrected image preprocessing transformations. The key addition is `transforms.Lambda(lambda img: img.convert("RGB"))`. This explicitly converts every image to a 3-channel RGB format, resolving the `RuntimeError` encountered previously with grayscale images. The rest of the transformations remain the same, including random augmentations for training and standard resizing/cropping for evaluation, followed by normalization.

In [ ]:
TEST 2

In [ ]:
from torchvision import transforms

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([

    # Force every image to RGB
    transforms.Lambda(lambda img: img.convert("RGB")),

    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=MEAN,
        std=STD
    )
])


eval_transform = transforms.Compose([

    # Force every image to RGB
    transforms.Lambda(lambda img: img.convert("RGB")),

    transforms.Resize(256),
    transforms.CenterCrop(224),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=MEAN,
        std=STD
    )
])

print("✅ RGB preprocessing ready")

✅ RGB preprocessing ready


### Corrected Dataset Setup (Attempt 2)
This section recreates the `Caltech256` datasets and `Subset` datasets for training, validation, and testing, but now using the `train_transform` and `eval_transform` from the corrected preprocessing step. This ensures that all images are consistently converted to RGB format before any further transformations, preventing data loading errors.

In [ ]:
from torch.utils.data import Subset
from torchvision.datasets import Caltech256

train_full_dataset = Caltech256(
    root=str(LOCAL_DATA_ROOT),
    download=False,
    transform=train_transform
)

eval_full_dataset = Caltech256(
    root=str(LOCAL_DATA_ROOT),
    download=False,
    transform=eval_transform
)

train_dataset = Subset(
    train_full_dataset,
    train_indices
)

val_dataset = Subset(
    eval_full_dataset,
    val_indices
)

test_dataset = Subset(
    eval_full_dataset,
    test_indices
)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

Train: 20846
Validation: 4467
Test: 4467


### Corrected DataLoader Creation (Attempt 2)
This cell recreates the `DataLoader` objects using the corrected `train_dataset`, `val_dataset`, and `test_dataset`. With the updated preprocessing transforms ensuring consistent image formats, these DataLoaders can now efficiently and correctly provide batches of images and labels without encountering the previous runtime errors.

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("✅ DataLoaders recreated")

✅ DataLoaders recreated


### Verify Corrected DataLoader Output
This cell tests the `train_loader` after applying the corrected preprocessing transforms and recreating the DataLoaders. It fetches the first batch of images and labels and prints their shapes and data types. This confirms that the data is now being loaded correctly, with images having the expected `[batch_size, 3, 224, 224]` shape (3 channels for RGB) and appropriate data types.

In [ ]:
images, labels = next(iter(train_loader))

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)

print("Images dtype:", images.dtype)
print("Labels dtype:", labels.dtype)

print("\nFirst 10 labels:")
print(labels[:10])

Images shape: torch.Size([32, 3, 224, 224])
Labels shape: torch.Size([32])
Images dtype: torch.float32
Labels dtype: torch.int64

First 10 labels:
tensor([ 44, 238,  91, 143, 158, 104, 225,  89, 144,  13])


### Data Loader Verification Function
This helper function, `check_loader`, is designed to thoroughly verify the integrity of the data provided by a `DataLoader`. It iterates through the entire loader, checking for:
1.  **Image Shape Consistency:** Ensures all images have the expected 4 dimensions and the correct `(channels, height, width)` format.
2.  **Label Range:** Confirms that the labels fall within the expected range of object classes.
3.  **Total Image Count:** Aggregates the total number of images processed, providing a final count for each split. This function provides confidence that the data is correctly prepared for model training.

In [ ]:
def check_loader(loader, name):
    total_images = 0
    min_label = 999999
    max_label = -1

    for images, labels in loader:
        # Check image format
        assert images.ndim == 4
        assert images.shape[1:] == (3, 224, 224)

        # Check labels
        min_label = min(min_label, labels.min().item())
        max_label = max(max_label, labels.max().item())

        total_images += images.size(0)

    print(f"✅ {name}")
    print("Images:", total_images)
    print("Label range:", min_label, "→", max_label)
    print()


check_loader(train_loader, "Train")
check_loader(val_loader, "Validation")
check_loader(test_loader, "Test")

✅ Train
Images: 20846
Label range: 0 → 255

✅ Validation
Images: 4467
Label range: 0 → 255

✅ Test
Images: 4467
Label range: 0 → 255



In [ ]:
import torch
import torch.nn as nn
from torchvision.models import resnet18

# Load ResNet-18 architecture without pretrained weights
model = resnet18(weights=None)

# Original final layer is for 1000 ImageNet classes
print("Original final layer:")
print(model.fc)

# Change output from 1000 classes to our 256 Caltech classes
model.fc = nn.Linear(
    model.fc.in_features,
    256
)

print("\nNew final layer:")
print(model.fc)

Original final layer:
Linear(in_features=512, out_features=1000, bias=True)

New final layer:
Linear(in_features=512, out_features=256, bias=True)


In [ ]:
print("Layer 1 blocks:", len(model.layer1))
print("Layer 2 blocks:", len(model.layer2))
print("Layer 3 blocks:", len(model.layer3))
print("Layer 4 blocks:", len(model.layer4))

Layer 1 blocks: 2
Layer 2 blocks: 2
Layer 3 blocks: 2
Layer 4 blocks: 2


In [ ]:
import torch.nn as nn
from torchvision.models.resnet import ResNet, BasicBlock

def build_resnet(blocks):
    model = ResNet(
        block=BasicBlock,
        layers=blocks,
        num_classes=256
    )
    return model

In [ ]:
import torch

test_model = build_resnet([2, 2, 2, 2])

print("Layer 1 blocks:", len(test_model.layer1))
print("Layer 2 blocks:", len(test_model.layer2))
print("Layer 3 blocks:", len(test_model.layer3))
print("Layer 4 blocks:", len(test_model.layer4))

print("\nFinal output classes:", test_model.fc.out_features)

# Fake image: batch of 1 RGB image, 224x224
dummy_image = torch.randn(1, 3, 224, 224)

with torch.no_grad():
    output = test_model(dummy_image)

print("Output shape:", output.shape)

Layer 1 blocks: 2
Layer 2 blocks: 2
Layer 3 blocks: 2
Layer 4 blocks: 2

Final output classes: 256
Output shape: torch.Size([1, 256])


In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

model_small = build_resnet([1, 1, 1, 1])
model_original = build_resnet([2, 2, 2, 2])

print("Small model:")
print("Blocks:", [1, 1, 1, 1])
print("Parameters:", count_parameters(model_small))

print("\nOriginal ResNet-18 style:")
print("Blocks:", [2, 2, 2, 2])
print("Parameters:", count_parameters(model_original))

Small model:
Blocks: [1, 1, 1, 1]
Parameters: 5037120

Original ResNet-18 style:
Blocks: [2, 2, 2, 2]
Parameters: 11307840


In [ ]:
# =========================================
# Track 1 - ResNet Depth Search Space
# Vary the number of BasicBlocks in each stage
# to change the total number of hidden layers.
# =========================================

SEARCH_SPACE = [
    [1, 1, 1, 1],  # Small / shallow ResNet
    [2, 2, 2, 2],  # Original ResNet-18
    [3, 3, 3, 3],  # Deeper ResNet
]

In [ ]:
[1,1,1,1]
4 BasicBlocks
8 Conv layers blocks
Parameters ≈ 5.0M

[2,2,2,2]
8 BasicBlocks
16 Conv layers blocks
Parameters ≈ 11.3M
ResNet-18

[3,3,3,3]
12 BasicBlocks
24 Conv layers blocks


SyntaxError: invalid character '≈' (U+2248) (223592722.py, line 4)

In [ ]:
# =========================================
# Verify all Search Space configurations
# =========================================

for blocks in SEARCH_SPACE:
    test_model = build_resnet(blocks)

    params = count_parameters(test_model)

    print("Blocks:", blocks)
    print("Parameters:", params)
    print("Output classes:", test_model.fc.out_features)
    print("-" * 40)

Blocks: [1, 1, 1, 1]
Parameters: 5037120
Output classes: 256
----------------------------------------
Blocks: [2, 2, 2, 2]
Parameters: 11307840
Output classes: 256
----------------------------------------
Blocks: [3, 3, 3, 3]
Parameters: 17578560
Output classes: 256
----------------------------------------


In [ ]:
# =========================================
# Sanity Check - One Training Step
# =========================================

import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Use original ResNet-18 configuration for the sanity check
sanity_model = build_resnet([2, 2, 2, 2]).to(device)

# Temporary loss and optimizer just to verify training works
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    sanity_model.parameters(),
    lr=1e-3
)

# Get one batch
images, labels = next(iter(train_loader))

images = images.to(device)
labels = labels.to(device)

# Training mode
sanity_model.train()

# Clear old gradients
optimizer.zero_grad()

# Forward pass
outputs = sanity_model(images)

# Calculate loss
loss = criterion(outputs, labels)

# Backpropagation
loss.backward()

# Update weights
optimizer.step()

print("Device:", device)
print("Images:", images.shape)
print("Labels:", labels.shape)
print("Outputs:", outputs.shape)
print("Loss:", loss.item())

print("\n✅ One training step completed successfully")

Device: cuda
Images: torch.Size([32, 3, 224, 224])
Labels: torch.Size([32])
Outputs: torch.Size([32, 256])
Loss: 5.636180877685547

✅ One training step completed successfully


In [ ]:
import time
import torch

# =========================================
# Train for one epoch
# =========================================

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    start_time = time.time()

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total
    epoch_time = time.time() - start_time

    return epoch_loss, epoch_accuracy, epoch_time


# =========================================
# Validation / Evaluation
# =========================================

def evaluate(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    start_time = time.time()

    with torch.no_grad():

        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            predictions = outputs.argmax(dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    loss = running_loss / total
    accuracy = correct / total
    eval_time = time.time() - start_time

    return loss, accuracy, eval_time


print("✅ Training and evaluation functions ready")

✅ Training and evaluation functions ready


In [ ]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Original ResNet-18 configuration
model = build_resnet([2, 2, 2, 2]).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

print("Starting 1 training epoch...")

train_loss, train_acc, train_time = train_one_epoch(
    model,
    train_loader,
    criterion,
    optimizer,
    device
)

print("\nStarting validation...")

val_loss, val_acc, val_time = evaluate(
    model,
    val_loader,
    criterion,
    device
)

print("\n=== Results ===")
print(f"Train Loss     : {train_loss:.4f}")
print(f"Train Accuracy : {train_acc * 100:.2f}%")
print(f"Train Time     : {train_time:.2f} sec")

print(f"\nVal Loss       : {val_loss:.4f}")
print(f"Val Accuracy   : {val_acc * 100:.2f}%")
print(f"Val Time       : {val_time:.2f} sec")

print("\n✅ One full epoch completed successfully")

Starting 1 training epoch...

Starting validation...

=== Results ===
Train Loss     : 5.2101
Train Accuracy : 5.89%
Train Time     : 97.04 sec

Val Loss       : 5.0119
Val Accuracy   : 7.05%
Val Time       : 21.56 sec

✅ One full epoch completed successfully


In [ ]:
# ============================================================
# Track 1 - ResNet Depth Experiment
# Ray Tune + MLflow + Colab-safe checkpoints
# ============================================================

import os
import gc
import random
import tempfile
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

import mlflow
import mlflow.pytorch

import ray
from ray import tune
from ray.tune import Checkpoint

# ------------------------------------------------------------
# Fixed experiment settings
# ------------------------------------------------------------

EXPERIMENT_EPOCHS = 10
LEARNING_RATE = 1e-3
FIXED_SEED = 42

RAY_STORAGE = PROJECT_ROOT / "ray_results"
RAY_STORAGE.mkdir(parents=True, exist_ok=True)

RAY_EXPERIMENT_NAME = "track1_resnet_depth_v2"

print("Ray storage:")
print(RAY_STORAGE)


# ------------------------------------------------------------
# Free GPU memory left from previous manual tests
# ------------------------------------------------------------

for variable_name in [
    "model",
    "sanity_model",
    "baseline_model",
    "test_model",
    "model_small",
    "model_original"
]:
    if variable_name in globals():
        del globals()[variable_name]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ------------------------------------------------------------
# Helper: actual serialized model size in KB
# ------------------------------------------------------------

def get_model_size_kb(model):

    with tempfile.NamedTemporaryFile(suffix=".pt") as tmp:
        torch.save(model.state_dict(), tmp.name)
        size_kb = os.path.getsize(tmp.name) / 1024

    return size_kb


# ------------------------------------------------------------
# Ray Trial
# ------------------------------------------------------------

def train_resnet_trial(config):

    # -------------------------
    # Fixed random seed
    # -------------------------

    random.seed(config["seed"])
    np.random.seed(config["seed"])
    torch.manual_seed(config["seed"])
    torch.cuda.manual_seed_all(config["seed"])


    blocks = config["blocks"]
    epochs = config["epochs"]
    lr = config["lr"]

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    print()
    print("=" * 60)
    print(f"🚀 TRIAL STARTED")
    print(f"Blocks: {blocks}")
    print(f"Device: {device}")
    print("=" * 60)


    # -------------------------
    # Build fresh model
    # -------------------------

    model = build_resnet(blocks).to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr
    )


    # -------------------------
    # Basic model information
    # -------------------------

    parameter_count = sum(
        p.numel() for p in model.parameters()
    )

    model_size_kb = get_model_size_kb(model)


    # -------------------------
    # Resume information
    # -------------------------

    start_epoch = 1
    best_val_accuracy = 0.0
    mlflow_run_id = None

    checkpoint = tune.get_checkpoint()

    if checkpoint is not None:

        with checkpoint.as_directory() as checkpoint_dir:

            checkpoint_path = os.path.join(
                checkpoint_dir,
                "checkpoint.pt"
            )

            state = torch.load(
                checkpoint_path,
                map_location=device,
                weights_only=False
            )

            model.load_state_dict(
                state["model_state_dict"]
            )

            optimizer.load_state_dict(
                state["optimizer_state_dict"]
            )

            start_epoch = state["epoch"] + 1

            best_val_accuracy = state[
                "best_val_accuracy"
            ]

            mlflow_run_id = state.get(
                "mlflow_run_id"
            )

        print(
            f"♻️ Resuming from epoch {start_epoch}"
        )


    # -------------------------
    # MLflow
    # -------------------------

    mlflow.set_tracking_uri(TRACKING_URI)

    mlflow.set_experiment(
        "caltech256_track1"
    )

    run_name = (
        "resnet_blocks_"
        + "_".join(map(str, blocks))
    )

    if mlflow_run_id is None:

        mlflow_context = mlflow.start_run(
            run_name=run_name
        )

    else:

        mlflow_context = mlflow.start_run(
            run_id=mlflow_run_id
        )


    with mlflow_context as active_run:

        # Log fixed configuration once
        if mlflow_run_id is None:

            mlflow.log_params({
                "blocks": str(blocks),
                "epochs": epochs,
                "optimizer": "Adam",
                "learning_rate": lr,
                "batch_size": train_loader.batch_size,
                "seed": config["seed"],
                "num_classes": 256,
                "pretrained": False,
                "parameter_count": parameter_count,
                "model_size_kb": model_size_kb,
            })


        # -------------------------
        # Epoch loop
        # -------------------------

        for epoch in range(
            start_epoch,
            epochs + 1
        ):

            print()
            print(
                f"Blocks {blocks} | "
                f"Epoch {epoch}/{epochs}"
            )


            # ---------------------
            # Training
            # ---------------------

            if torch.cuda.is_available():
                torch.cuda.reset_peak_memory_stats()

            train_loss, train_acc, train_time = (
                train_one_epoch(
                    model,
                    train_loader,
                    criterion,
                    optimizer,
                    device
                )
            )

            if torch.cuda.is_available():

                train_memory_kb = (
                    torch.cuda.max_memory_allocated()
                    / 1024
                )

            else:
                train_memory_kb = 0


            # ---------------------
            # Validation
            # ---------------------

            if torch.cuda.is_available():
                torch.cuda.reset_peak_memory_stats()

            val_loss, val_acc, val_time = evaluate(
                model,
                val_loader,
                criterion,
                device
            )

            if torch.cuda.is_available():

                inference_memory_kb = (
                    torch.cuda.max_memory_allocated()
                    / 1024
                )

            else:
                inference_memory_kb = 0


            best_val_accuracy = max(
                best_val_accuracy,
                val_acc
            )


            # ---------------------
            # Metrics
            # ---------------------

            metrics = {

                "train_loss":
                    train_loss,

                "train_accuracy":
                    train_acc,

                "val_loss":
                    val_loss,

                "val_accuracy":
                    val_acc,

                "best_val_accuracy":
                    best_val_accuracy,

                "train_time_sec":
                    train_time,

                "val_time_sec":
                    val_time,

                "train_peak_gpu_memory_kb":
                    train_memory_kb,

                "inference_peak_gpu_memory_kb":
                    inference_memory_kb,

                "model_size_kb":
                    model_size_kb,
            }


            # ---------------------
            # MLflow metrics
            # ---------------------

            mlflow.log_metrics(
                metrics,
                step=epoch
            )


            print(
                f"Train Acc: "
                f"{train_acc * 100:.2f}%"
            )

            print(
                f"Val Acc  : "
                f"{val_acc * 100:.2f}%"
            )


            # ---------------------
            # Checkpoint every epoch
            # ---------------------

            with tempfile.TemporaryDirectory() as tmpdir:

                checkpoint_path = os.path.join(
                    tmpdir,
                    "checkpoint.pt"
                )

                torch.save({

                    "epoch":
                        epoch,

                    "model_state_dict":
                        model.state_dict(),

                    "optimizer_state_dict":
                        optimizer.state_dict(),

                    "best_val_accuracy":
                        best_val_accuracy,

                    "blocks":
                        blocks,

                    "mlflow_run_id":
                        active_run.info.run_id,

                }, checkpoint_path)


                ray_checkpoint = (
                    Checkpoint.from_directory(
                        tmpdir
                    )
                )


                # Report to Ray + save checkpoint
                tune.report(
                    metrics,
                    checkpoint=ray_checkpoint
                )


        # -------------------------
        # Save final model to MLflow
        # -------------------------

        mlflow.pytorch.log_model(
            model,
            name="model",
            serialization_format="pickle"
        )


    print(
        f"✅ Finished blocks {blocks}"
    )


print("✅ Ray training function is ready")

Ray storage:
/content/drive/MyDrive/vector_ai_track1/ray_results
✅ Ray training function is ready


In [ ]:
# ============================================================
# START / RESUME TRACK 1 - ResNet Depth Experiment
# ============================================================

import ray
from ray import tune

# Restart Ray cleanly
if ray.is_initialized():
    ray.shutdown()

ray.init(ignore_reinit_error=True)


# Give each trial the full T4 GPU
ray_trainable = tune.with_resources(
    train_resnet_trial,
    resources={
        "cpu": 2,
        "gpu": 1
    }
)


experiment_path = str(
    RAY_STORAGE / RAY_EXPERIMENT_NAME
)


# ============================================================
# Resume experiment if Colab disconnected previously
# ============================================================

if tune.Tuner.can_restore(experiment_path):

    print("♻️ Existing experiment found.")
    print("Resuming experiment...")

    tuner = tune.Tuner.restore(
        experiment_path,
        trainable=ray_trainable,
        resume_unfinished=True,
        resume_errored=True
    )


# ============================================================
# Start a new experiment
# ============================================================

else:

    print("🚀 Starting NEW Track 1 experiment")

    tuner = tune.Tuner(

        ray_trainable,

        param_space={
            "blocks": tune.grid_search(SEARCH_SPACE),
            "epochs": 10,
            "lr": 0.001,
            "seed": 42,
        },

        tune_config=tune.TuneConfig(
            # One GPU -> run one model at a time
            max_concurrent_trials=1
        ),

        run_config=tune.RunConfig(
            name=RAY_EXPERIMENT_NAME,
            storage_path=str(RAY_STORAGE),

            checkpoint_config=tune.CheckpointConfig(
                num_to_keep=2
            ),

            failure_config=tune.FailureConfig(
                max_failures=2
            ),
        )
    )


print()
print("========================================")
print("🚀 TRACK 1 TRAINING STARTING")
print("========================================")
print()


results = tuner.fit()

2026-09-24 18:54:14,812	INFO worker.py:2024 -- Started a local Ray instance.
2026-09-24 18:54:19,442	INFO tune_controller.py:448 -- Restoring the run from the latest experiment state file: experiment_state-2026-09-24_18-29-40.json


♻️ Existing experiment found.
Resuming experiment...

🚀 TRACK 1 TRAINING STARTING

+-----------------------------------------------------------+
| Configuration for experiment     track1_resnet_depth_v2   |
+-----------------------------------------------------------+
| Search algorithm                 BasicVariantGenerator    |
| Scheduler                        FIFOScheduler            |
| Number of trials                 3                        |
+-----------------------------------------------------------+

View detailed results here: /content/drive/MyDrive/vector_ai_track1/ray_results/track1_resnet_depth_v2
To visualize your results with TensorBoard, run: `tensorboard --logdir /tmp/ray/session_2026-09-24_18-54-03_089259_543/artifacts/2026-09-24_18-27-56/track1_resnet_depth_v2/driver_artifacts`

Trial status: 3 PENDING
Current time: 2026-09-24 18:54:19. Total running time: 0s
Logical resource usage: 2.0/2 CPUs, 1.0/1 GPUs (0.0/1.0 accelerator_type:T4)
+----------------------------

(train_resnet_trial pid=31766) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/content/drive/MyDrive/vector_ai_track1/ray_results/track1_resnet_depth_v2/train_resnet_trial_a9809_00000_0_blocks=1_1_1_1_2026-09-24_18-27-56/checkpoint_000000)


(train_resnet_trial pid=31766) 
(train_resnet_trial pid=31766) Blocks [1, 1, 1, 1] | Epoch 2/10
Trial status: 1 RUNNING | 2 PENDING
Current time: 2026-09-24 18:56:50. Total running time: 2min 30s
Logical resource usage: 2.0/2 CPUs, 1.0/1 GPUs (0.0/1.0 accelerator_type:T4)
+---------------------------------------------------------------------------------------------------------------------------------------+
| Trial name                       status       iter     total time (s)     train_loss     train_accuracy     val_loss     val_accuracy |
+---------------------------------------------------------------------------------------------------------------------------------------+
| train_resnet_trial_a9809_00000   RUNNING         1            118.334        5.06932          0.0745467      4.65306         0.113275 |
| train_resnet_trial_a9809_00001   PENDING                                                                                              |
| train_resnet_trial_a9809_00002   PE

(train_resnet_trial pid=31766) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/content/drive/MyDrive/vector_ai_track1/ray_results/track1_resnet_depth_v2/train_resnet_trial_a9809_00000_0_blocks=1_1_1_1_2026-09-24_18-27-56/checkpoint_000001)


(train_resnet_trial pid=31766) 
(train_resnet_trial pid=31766) Blocks [1, 1, 1, 1] | Epoch 3/10
Trial status: 1 RUNNING | 2 PENDING
Current time: 2026-09-24 18:58:50. Total running time: 4min 30s
Logical resource usage: 2.0/2 CPUs, 1.0/1 GPUs (0.0/1.0 accelerator_type:T4)
+---------------------------------------------------------------------------------------------------------------------------------------+
| Trial name                       status       iter     total time (s)     train_loss     train_accuracy     val_loss     val_accuracy |
+---------------------------------------------------------------------------------------------------------------------------------------+
| train_resnet_trial_a9809_00000   RUNNING         2            232.852        4.66127           0.109469      4.44492         0.132304 |
| train_resnet_trial_a9809_00001   PENDING                                                                                              |
| train_resnet_trial_a9809_00002   PE

... (58 outputs left out)

(train_resnet_trial pid=43248) 2026/09/24 19:58:02 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.26.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torchvision==0.26.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.



Trial train_resnet_trial_a9809_00002 completed after 10 iterations at 2026-09-24 19:58:02. Total running time: 1hr 3min 43s
+-------------------------------------------------------------------+
| Trial train_resnet_trial_a9809_00002 result                       |
+-------------------------------------------------------------------+
| checkpoint_dir_name                             checkpoint_000009 |
| time_this_iter_s                                        134.86428 |
| time_total_s                                           1339.76139 |
| training_iteration                                             10 |
| best_val_accuracy                                         0.32147 |
| inference_peak_gpu_memory_kb                             528451.5 |
| model_size_kb                                         68774.72949 |
| train_accuracy                                            0.28624 |
| train_loss                                                3.34474 |
| train_peak_gpu_memory_kb         

2026-09-24 19:58:03,151	INFO tune.py:1007 -- Wrote the latest version of all result files and experiment state to '/content/drive/MyDrive/vector_ai_track1/ray_results/track1_resnet_depth_v2' in 0.2282s.



Trial status: 3 TERMINATED
Current time: 2026-09-24 19:58:03. Total running time: 1hr 3min 43s
Logical resource usage: 2.0/2 CPUs, 1.0/1 GPUs (0.0/1.0 accelerator_type:T4)
+-----------------------------------------------------------------------------------------------------------------------------------------+
| Trial name                       status         iter     total time (s)     train_loss     train_accuracy     val_loss     val_accuracy |
+-----------------------------------------------------------------------------------------------------------------------------------------+
| train_resnet_trial_a9809_00000   TERMINATED       10            1149.55        3.24792           0.304183      3.07571         0.331542 |
| train_resnet_trial_a9809_00001   TERMINATED       10            1246.62        3.30037           0.295692      3.06196         0.3349   |
| train_resnet_trial_a9809_00002   TERMINATED       10            1339.76        3.34474           0.286242      3.13756       